E-21: IHDP semi-synthetic data (illustration, I-type; statements S1 and S2)

Design: `doc/2026-10-07_experiment_registration.md` (parent repository), E-21, §1.4 and §1.5. Under U28 the design document is not binding; the choices it leaves open are in the parent repository's `docs/spec/02_data-and-experiments.md` §2.1 ("E-21 の" rows). One notebook for every stage; the runner passes the stage:

- `python3 notebooks/experiments/run_registered.py E-21 stage0.5 18_E21_ihdp.ipynb`
- `... E-21 stage1 ...` (after the pilot is committed).

Data, arms, the replication function and the aggregation are in `grrexp/e21.py`. The data hashes are checked by the run recorder before any replication (§1.7). A solver that raises stops the notebook and the run is recorded as failed (§1.1).

In [ ]:
import io
import json
import os
import time

import pandas as pd

from grrexp import env, outputs

gr = env.use_submodule_genriesz()
from grrexp import e21, parallel  # noqa: E402
from grrexp.seeds import CONFIRMATORY_ENTROPY, PILOT_ENTROPY, Seeds  # noqa: E402

STAGE = json.loads(os.environ[outputs.RUN_ENV])["stage"]
rec = outputs.RunRecorder(21, STAGE, cells=e21.CELLS, arms=[e21.record_label(a) for a in e21.ARM_LABELS])
#: Stage 1 workers (results do not depend on the number of workers; Stage 0.5 checks 1 vs 12)
STAGE1_WORKERS = 6
STAGE

# The workflow after the replications (shared by Stage 0.5 and Stage 1)

The EB check summary (every EB fold fit was checked against the independent dual Newton when it was fitted; a difference above the tolerance stops the run), the per-arm summary (bias, SD and RMSE of the error against the replication's true value over successful replications; unconditional coverage and coverage among successes; failure rate and statuses), statement S2 (|c - 0.95| <= 0.044), and statement S1 (paired bootstrap, stream 4, family 1, 10^4 draws, per GRR arm in `e21.ARM_LABELS` order).

In [ ]:
def aggregate(raw, seeds):
    eb = e21.eb_summary(raw)
    summ = e21.summarise(raw)
    s1 = e21.s1_intervals(raw, seeds)
    summ["eb_check"] = json.dumps(eb)
    summ["s1"] = json.dumps(s1)
    return summ, s1

# Stage 0.5: timing of the complete workflow, status counts, 1-versus-12-worker identity (no estimates kept)

In [ ]:
if STAGE == "stage0.5":
    parallel.configure_worker()
    timing, serial, para, all_tasks = [], [], [], []
    for workers in (1, outputs.PILOT_WORKERS):
        for ci, cell in enumerate(e21.CELLS):
            t0 = time.perf_counter()
            tasks = e21.tasks_for(PILOT_ENTROPY, outputs.PILOT_REPS, cells=[ci])
            res = parallel.run_tasks(e21.replicate, tasks, workers)
            aggregate(e21.raw_frame(tasks, res), Seeds(21, entropy=PILOT_ENTROPY))
            timing.append({"cell": cell, "workers": workers, "seconds": time.perf_counter() - t0})
            if workers == 1:
                serial += res
                all_tasks += tasks
            else:
                para += res
    digest = parallel.assert_bit_identical(serial, para, expected=len(all_tasks))
    raw = e21.raw_frame(all_tasks, serial)
    t0 = time.perf_counter()
    summ, s1 = aggregate(raw, Seeds(21, entropy=PILOT_ENTROPY))
    buf = io.StringIO()
    summ.to_csv(buf, index=False)
    buf.seek(0)
    e21.tables(pd.read_csv(buf), s1)
    whole_seconds = time.perf_counter() - t0
    arms = raw[raw["arm"] != "outcome"]
    counts = arms.groupby(["cell", "arm", "status"]).size().reset_index(name="count")
    status_df = pd.DataFrame({"cell": [e21.CELLS[c] for c in counts["cell"]],
                              "arm": counts["arm"].map(e21.record_label),
                              "status": counts["status"], "count": counts["count"].astype(int)})
    rec.write_pilot("timing.csv", pd.DataFrame(timing))
    rec.write_pilot("status_counts.csv", status_df)
    rec.record_parallel_identity(cells=e21.CELLS, reps=outputs.PILOT_REPS, n_jobs=outputs.PILOT_WORKERS,
                                 digest=digest)
    print(pd.DataFrame(timing).groupby("workers")["seconds"].sum(), whole_seconds)
    print(status_df.groupby(["arm", "status"])["count"].sum().to_string())
    rec.finalize(R={"main": outputs.PILOT_REPS}, n={"main": [e21.N]},
                 warnings=e21.total_warnings(raw), failures=int((arms["status"] != "ok").sum()),
                 extra={"whole_pilot_aggregation_and_table_seconds": whole_seconds})
    del raw

# Stage 1: all 100 replications

In [ ]:
if STAGE == "stage1":
    parallel.configure_worker()
    tasks = e21.tasks_for(CONFIRMATORY_ENTROPY, e21.R)
    t0 = time.perf_counter()
    results = parallel.run_tasks(e21.replicate, tasks, STAGE1_WORKERS)
    print("seconds", time.perf_counter() - t0)
    raw = e21.raw_frame(tasks, results)
    rec.write_raw(raw)
    n_warn = e21.total_warnings(raw)
    raw.shape, n_warn

In [ ]:
if STAGE == "stage1":
    summ, s1 = aggregate(raw, Seeds(21))
    rec.write_summary(summ)
    json.loads(summ["eb_check"].iloc[0]), s1

# Tables and macros (from summary.csv only, §4)

In [ ]:
if STAGE == "stage1":
    S = outputs.read_summary(rec)
    s1 = {k: tuple(v) for k, v in json.loads(S["s1"].iloc[0]).items()}
    tabs, macros, statements = e21.tables(S, s1)
    for name, tex in tabs.items():
        rec.write_table(name, tex)
    rec.write_macros(macros)
    print(statements)
    arms = raw[raw["arm"] != "outcome"]
    rec.finalize(R={"main": e21.R}, n={"main": [e21.N]}, warnings=int(n_warn),
                 failures=int((arms["status"] != "ok").sum()))